# Build Your Own Search Engine

A step-by-step implementation of a search engine over FAQ documents,
progressing from keyword matching to semantic embeddings.

Based on the [Build Your Own Search Engine](https://aishippinglabs.com/workshops/2026-05-14-build-your-own-search-engine)
workshop by Alexey Grigorev / AI Shipping Labs.

Dataset: DataTalks.Club Zoomcamp FAQ (402 documents).

In [ ]:
import requests
import pandas as pd

documents = requests.get(
    'https://datatalks.club/faq/json/data-engineering-zoomcamp.json'
).json()

df = pd.DataFrame(documents)
df = df.rename(columns={'answer': 'text'})
df.head()

## Dataset overview

In [ ]:
print(f"Total documents: {len(df)}")
print(f"\nUnique sections:")
print(df['section'].value_counts())

## Part 1: Text Search and TF-IDF

### Bag of Words — CountVectorizer

Before working with the full dataset, we use 5 short sentences to understand
what vectorization does. Each word becomes a column, each document a row.
The number indicates how many times that word appears in that document.

Words like "on", "and", "for" disappear — those are stopwords, filtered out
because they carry no search value.

In [ ]:
docs_example = [
    "Course starts on 15th Jan 2024",
    "Prerequisites listed on GitHub",
    "Submit homeworks after start date",
    "Registration not required for participation",
    "Setup Google Cloud and Python before course",
]

from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english')
X = cv.fit_transform(docs_example)
names = cv.get_feature_names_out()

df_docs = pd.DataFrame(X.toarray(), columns=names).T
df_docs

### TF-IDF — TfidfVectorizer

Raw counts have a problem: a long document mentioning "course" 10 times
always beats a short one mentioning it once, even if the short one is
more relevant.

TF-IDF fixes this by weighting words by how rare they are across all
documents. A word like "github" that appears in only one document gets
a higher weight than "course" which appears in many.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

cv = TfidfVectorizer(stop_words='english')
X = cv.fit_transform(docs_example)

names = cv.get_feature_names_out()

df_docs = pd.DataFrame(X.toarray(), columns=names).T
df_docs.round(2)

### Vectorizing a query

To search, we convert the query into a vector using the **same vocabulary**
the documents used. We use `transform` (not `fit_transform`) because we don't
want to learn a new vocabulary — we want to reuse the one already learned from
the documents.

In [ ]:
query = "Do I need to know python to sign up for the January course?"

q = cv.transform([query])
q.toarray()

### Exploring the internal vocabulary

`cv.vocabulary_` shows the dictionary the vectorizer learned: each word
and its position in the vector. Words from the query not in this vocabulary
(like "need", "know", "sign") are silently ignored — a key limitation of TF-IDF.

In [ ]:
cv.vocabulary_

### Dot product vs cosine similarity

Before using `cosine_similarity`, we can compute similarity manually via
dot product. With TF-IDF vectors (which are already normalized to length 1),
both produce identical results — dividing by 1 changes nothing.

In [ ]:
X.dot(q.T).toarray()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity(X, q)

### Vectorizing the full FAQ dataset

Now we apply TF-IDF to the real dataset. We vectorize three fields separately
because each carries different information. `min_df=3` removes words that
appear in fewer than 3 documents — too rare to be useful for search.

In [ ]:
fields = ['section', 'question', 'text']
transformers = {}
matrices = {}

for field in fields:
    cv = TfidfVectorizer(stop_words='english', min_df=3)
    X = cv.fit_transform(df[field])

    transformers[field] = cv
    matrices[field] = X

### Vocabulary size per field

Each field learned a different vocabulary. `section` has few unique words
(short, repetitive titles). `text` has the largest vocabulary (long, varied answers).

In [ ]:
for field in fields:
    print(f"{field}: {len(transformers[field].vocabulary_)} words")

### Matrix dimensions

Each matrix has shape `(documents, vocabulary_size)`. 402 rows — one per FAQ entry.
Columns vary by field depending on how rich the vocabulary is.

In [ ]:
print("Matrix dimensions (Documents, Vocabulary size):")
for field in fields:
    print(f"{field}: {matrices[field].shape}")

### Searching over a single field

First attempt: search only over `text` (the answers). Results are relevant
but noisy — user queries are phrased more like questions than answers.

In [ ]:
query = "I just signed up. Is it too late to join the course?"

q = transformers['text'].transform([query])
score = cosine_similarity(matrices['text'], q).flatten()

### Filtering by course

We zero out scores from other courses using a boolean mask.
`True * score = score`, `False * score = 0` — documents from other courses
silently drop to the bottom of the ranking.

In [ ]:
import numpy as np

mask = (df.course == 'data-engineering-zoomcamp').values
score = score * mask

In [ ]:
idx = np.argsort(-score)[:10]
df.iloc[idx].text

## Part 2: Boosting, Filtering and TextSearch Class

### Field boosting

Searching only `text` misses relevant documents. User queries are phrased
like questions, so the `question` field is a better match. We give it 3x
weight — its score contribution is tripled before accumulating.

The maximum possible score is `3.0 + 1.0 + 1.0 = 5.0` (perfect match on all three fields).

In [ ]:
query = "I just signed up. Is it too late to join the course?"
boost = {'question': 3.0}
score = np.zeros(len(df))

for f in fields:
    b = boost.get(f, 1.0)
    q = transformers[f].transform([query])
    s = cosine_similarity(matrices[f], q).flatten()
    score = score + b * s

idx = np.argsort(-score)[:5]
print(score[idx])

### Inspecting top results with boosting

Printing scores and snippets to see the effect of boosting `question` 3x.

In [ ]:
# Apply course filter
mask = (df.course == 'data-engineering-zoomcamp').values
score = score * mask

# Sort descending and take top 5
idx = np.argsort(-score)[:5]

print("Top 5 results with boosting on 'question':\n")
for i, index in enumerate(idx):
    print(f"--- Rank {i+1} (Score: {score[index]:.3f}) ---")
    print(f"Question: {df.iloc[index]['question']}")
    print(f"Text: {df.iloc[index]['text'][:100]}...\n")

### Using a filters dictionary

Refactoring the filter into a dictionary — the same pattern used in the
`TextSearch` class. Cleaner and extensible to multiple filter fields.

In [ ]:
filters = {
    'course': 'data-engineering-zoomcamp',
}

for field, value in filters.items():
    mask = (df[field] == value).values
    score = score * mask

idx = np.argsort(-score)[:10]
results = df.iloc[idx]
results.to_dict(orient='records')

### TextSearch class — putting it all together

We encapsulate the full pipeline (vectorize, boost, filter, rank) into a
reusable class. This became the `minsearch` library used in the LLM Zoomcamp.

`fit` is called once (expensive — vectorizes all documents).
`search` is called per query (fast — only vectorizes the query).

The production version with type hints and docstrings lives in `search_engine.py`.

In [ ]:
class TextSearch:

    def __init__(self, text_fields):
        self.text_fields = text_fields
        self.matrices = {}
        self.vectorizers = {}

    def fit(self, records, vectorizer_params={}):
        self.df = pd.DataFrame(records)
        if 'answer' in self.df.columns:
            self.df = self.df.rename(columns={'answer': 'text'})

        for f in self.text_fields:
            cv = TfidfVectorizer(**vectorizer_params)
            X = cv.fit_transform(self.df[f])
            self.matrices[f] = X
            self.vectorizers[f] = cv

    def search(self, query, n_results=10, boost={}, filters={}):
        score = np.zeros(len(self.df))

        for f in self.text_fields:
            b = boost.get(f, 1.0)
            q = self.vectorizers[f].transform([query])
            s = cosine_similarity(self.matrices[f], q).flatten()
            score = score + b * s

        for field, value in filters.items():
            mask = (self.df[field] == value).values
            score = score * mask

        idx = np.argsort(-score)[:n_results]
        results = self.df.iloc[idx]
        return results.to_dict(orient='records')

### Using TextSearch

In [ ]:
# Instantiate the engine with the fields to search over
index = TextSearch(text_fields=['section', 'question', 'text'])

# Fit: learns vocabulary and vectorizes all documents
index.fit(documents, vectorizer_params={'stop_words': 'english', 'min_df': 3})

# Search: vectorizes the query and returns top results
results = index.search(
    query='I just signed up. Is it too late to join the course?',
    n_results=5,
    boost={'question': 3.0},
    filters={'course': 'data-engineering-zoomcamp'}
)
print(results)

## Part 3: SVD and NMF Embeddings

TF-IDF misses synonyms — "enroll" and "register" score 0 similarity despite
meaning the same thing. Embeddings fix this by representing documents as
dense vectors where meaning determines proximity, not exact words.

### SVD — Singular Value Decomposition

SVD compresses the 1352-dimensional TF-IDF matrix down to 16 dimensions,
capturing "latent topics" — groups of words that tend to appear together.

Limitation: produces negative values, making dimensions hard to interpret.
Also still bag-of-words underneath — no understanding of word order.

In [ ]:
from sklearn.decomposition import TruncatedSVD

X = matrices['text']
cv = transformers['text']

svd = TruncatedSVD(n_components=16)
X_emb = svd.fit_transform(X)
X_emb[0]

### Vectorizing the query with SVD

We use `cv.transform` (not `fit_transform`) to reuse the existing vocabulary,
then `svd.transform` to project into the same 16-dimensional space as the documents.
Both query and documents must live in the same space for cosine similarity to work.

In [ ]:
query = 'I just signed up. Is it too late to join the course?'

Q = cv.transform([query])
Q_emb = svd.transform(Q)
Q_emb[0]

In [ ]:
score = cosine_similarity(X_emb, Q_emb).flatten()
idx = np.argsort(-score)[:10]
list(df.loc[idx].text)

### NMF — Non-Negative Matrix Factorization

Similar to SVD but constrained to non-negative values. Each dimension can be
interpreted as a concrete topic (no negative "anti-topics"). Tends to produce
sparse representations — most documents belong strongly to just a few topics.

Results are similar to SVD on this small dataset. Both struggle because 16
dimensions and 402 documents aren't enough to learn meaningful semantics.

In [ ]:
from sklearn.decomposition import NMF

nmf = NMF(n_components=16)
X_emb = nmf.fit_transform(X)
X_emb[0]

In [ ]:
Q = cv.transform([query])
Q_emb = nmf.transform(Q)
Q_emb[0]

score = cosine_similarity(X_emb, Q_emb).flatten()
idx = np.argsort(-score)[:10]
list(df.loc[idx].text)

## Part 4: BERT Embeddings

SVD and NMF ignore word order entirely. BERT is a transformer model that reads
text sequentially and produces embeddings that capture both meaning and context.

Key difference: *"I didn't like it"* and *"I liked it"* produce very different
BERT embeddings despite sharing most words. SVD/NMF can't distinguish them.

Trade-off: BERT takes 10+ minutes on CPU vs milliseconds for TF-IDF.

### Loading the tokenizer and model

The **tokenizer** converts text into integer token IDs.
The **model** converts those tokens into 768-dimensional vectors.
`model.eval()` switches to inference mode — no weight updates, just predictions.

In [ ]:
import torch
from transformers import BertModel, BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval()

### Tokenizing a small example

Before processing 402 documents, we inspect what the tokenizer produces.

- `input_ids`: each word converted to a numeric token ID. `101` = `[CLS]` (start), `102` = `[SEP]` (end), `0` = padding.
- `attention_mask`: `1` = real token, `0` = padding to ignore. BERT uses this to skip padded positions.
- `token_type_ids`: used when comparing sentence pairs. All `0` here since we're encoding single sentences.

In [ ]:
texts = [
    "Yes, we will keep all the materials after the course finishes.",
    "You can follow the course at your own pace after it finishes",
]

encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
encoded_input

### Running the model — mean pooling

`torch.no_grad()` disables gradient tracking — we're doing inference, not training.
`last_hidden_state` has shape `(sentences, tokens, 768)`. We average across the
token dimension (`dim=1`) to get one 768-dimensional vector per sentence.

In [ ]:
with torch.no_grad():
    outputs = model(**encoded_input)
    hidden_states = outputs.last_hidden_state

sentence_embeddings = hidden_states.mean(dim=1)
sentence_embeddings.shape

### Processing documents in batches

Computing embeddings one document at a time is slow. We split the 402 documents
into batches of 4 (safe for CPU memory) and process each batch together.

`make_batches` is extracted into `utils.py` for reuse across the project.

In [ ]:
def make_batches(seq, n):
    result = []
    for i in range(0, len(seq), n):
        batch = seq[i:i+n]
        result.append(batch)
    return result

In [ ]:
from tqdm.auto import tqdm

texts = df['text'].tolist()
text_batches = make_batches(texts, 4)

all_embeddings = []

for batch in tqdm(text_batches):
    encoded_input = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')

    with torch.no_grad():
        outputs = model(**encoded_input)
        hidden_states = outputs.last_hidden_state
        batch_embeddings = hidden_states.mean(dim=1)
        batch_embeddings_np = batch_embeddings.cpu().numpy()
        all_embeddings.append(batch_embeddings_np)

X_emb = np.vstack(all_embeddings)
X_emb.shape

### Searching with BERT embeddings over `text`

Same cosine similarity approach as TF-IDF, but now over 768-dimensional
semantic vectors. Results improve over SVD/NMF but are still noisy —
searching over `text` (answers) doesn't match the user's query language well.

In [ ]:
query = "I just signed up. Is it too late to join the course?"

encoded_q = tokenizer([query], padding=True, truncation=True, return_tensors='pt')

with torch.no_grad():
    outputs = model(**encoded_q)
    Q_emb = outputs.last_hidden_state.mean(dim=1).numpy()

score = cosine_similarity(X_emb, Q_emb).flatten()
idx = np.argsort(-score)[:10]
list(df.loc[idx].text)

### Exploring: BERT embeddings over `question` field

Observation: user queries are phrased like questions, not like answers.
Searching over `question` instead of `text` should produce better results.

This is an exploratory experiment — not part of the original workshop flow.

In [ ]:
# Compute BERT embeddings over the 'question' field
q_texts = df['question'].tolist()
q_batches = make_batches(q_texts, 4)
q_embeddings = []

for batch in tqdm(q_batches):
    enc = tokenizer(batch, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        out = model(**enc)
        emb = out.last_hidden_state.mean(dim=1).numpy()
        q_embeddings.append(emb)

Q_emb_docs = np.vstack(q_embeddings)

### Exploring: search over `question` with course filter

Combining BERT embeddings over `question` with a course filter.
Result: significantly better than searching over `text` — semantically
similar questions rank higher despite no keyword overlap.

In [ ]:
query = "I just signed up. Is it too late to join the course?"

encoded_q = tokenizer([query], padding=True, truncation=True, return_tensors='pt')
with torch.no_grad():
    outputs = model(**encoded_q)
    query_emb = outputs.last_hidden_state.mean(dim=1).numpy()

score = cosine_similarity(Q_emb_docs, query_emb).flatten()

# Apply course filter
mask = (df.course == 'data-engineering-zoomcamp').values
score = score * mask

idx = np.argsort(-score)[:5]
df.iloc[idx][['question', 'text']]

### Exploring: query vs questions (no filter)

Comparing the query embedding directly against all FAQ question embeddings,
without any course filter. BERT correctly retrieves *"Can I still join the
course after the start date?"* at position 9 — despite sharing no keywords
with the query.

In [ ]:
# Compare query embedding against all FAQ question embeddings
score = cosine_similarity(Q_emb_docs, Q_emb).flatten()
idx = np.argsort(-score)[:10]

# Print top matching questions
list(df.loc[idx].question)